# Reviews, screenplays and summaries in an embedding space: do they line up with profit?

Each film's text is turned into one vector, so that films whose text says similar things sit close together. Then we
ask whether films that sit close together also have similar money outcomes. Three texts, each with a different
relationship to the outcome:

| Text | Source | Written | What it could know about money |
|---|---|---|---|
| Audience reviews | MovieLens Tag Genome 2021 (IMDb reviews, nothing after 2017) | after release | the outcome itself ("it flopped"), hype, sequel status |
| Screenplays | MovieSum | before release | only what is on the page |
| Plot summaries | MovieSum (from Wikipedia) | after release, about the story | little, once names are removed |

Profitability is ln(worldwide gross / budget), revenue is ln(gross). Neither is real profit: marketing isn't in the
budget and cinemas keep part of the gross. The sample is the clean theatrical rule used everywhere in the repo
(USD budget and gross, 1975 to 2025, no rentals, regional or direct-to-video grosses, no streaming distributors).

Run order, from the repo root: `python -m analysis.embeddings.s01_corpora` (clean), `s02_embed` (vectors),
`s03_correlate` (tests), then this notebook. Every parameter is in `analysis/embeddings/config.yaml`; the two
added after the first results are under `post_hoc` and are labelled wherever they're used.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO))
import pandas as pd
from IPython.display import Markdown, display

from analysis.embeddings import report as R

pd.set_option("display.max_columns", 30, "display.width", 200)
res = R.load()
r = res["results"]
say = R.reading(res)
print(f"{len(r):,} measurements")

178 measurements


## 1. Cleaning

A language model reads everything in the text, including things that aren't the content. What each rule removes
and why:

- **Markup, HTML entities, URLs, tokeniser spacing** (MovieSum stores "did n't"): how the text was stored.
- **Review sentences about money** ("flopped at the box office", "$200 million"): a review is written after
  release, so this is the outcome leaking into the predictor. We drop the sentence, not the review.
- **The film's title, its cast and director (reviews), character names (screenplays) and every person's name
  (summaries, with spaCy NER)**: the model has read about famous films and stars, so with names left in, a vector
  can encode *which* film or *who* made it rather than what the text says. Who made it is the network half of the
  project; here we want the words.
- **Screenplay scene headings, speaker labels, parentheticals and transitions**: formatting.
- **Duplicate, very short and non-English reviews.** Each film gets 30 reviews sampled at random (seeded), so long
  runs of one reviewer or the file's order don't decide which ones we read.

Character names in screenplays are found without a model: a word from a speaker label counts as a name if the
script almost never writes it in lower case ("MARLA" is a name, "DOCTOR" isn't, because "the doctor" appears in
the descriptions). spaCy's PERSON tags are the second method; their agreement is below.

In [2]:
display(R.cleaning_table(res["cleaning"]))
chk = res["cleaning"]["screenplays"]["name_rule_vs_spacy"]
print(f"Name rule vs spaCy PERSON on {chk['scripts']} scripts x {chk['words_per_script']:,} words: "
      f"spaCy also tags {chk['share_of_rule_words_spacy_also_tags']:.0%} of the words the rule masks, and the rule "
      f"masks {chk['share_of_spacy_person_words_the_rule_masks']:.0%} of the words spaCy tags.")

,corpus,films,text kept,what cleaning did
0,Audience reviews,5405,"160,532 reviews, 30 per film (median)","3,788 duplicates, 1,686 too short and 297 non-..."
1,Screenplays,1630,"20,752 words per script (median)","9.6% of words removed (headings, speaker label..."
2,Plot summaries,1625,654 words per summary (median),"66,880 person names masked with spaCy NER (6.4..."


Name rule vs spaCy PERSON on 40 scripts x 3,000 words: spaCy also tags 53% of the words the rule masks, and the rule masks 72% of the words spaCy tags.


Reading the disagreements by hand: most words only the rule masks are character names spaCy misses (invented
names such as Linguini or Balian), with a few false ones (SERGEANT, Jerusalem). Most words only spaCy tags are
ordinary names that never speak (Jack, Steve) and some mistakes (Outside, Hesitates). Neither is ground truth;
both leave a small residue of names.

**Two problems found after the first results** (fixed, and marked post hoc in the config):
- OCR damage. Some MovieSum scripts were scanned and read badly ("gest:ure", "t~e"). The first word list for the
  "more profitable than expected" screenplays was entirely such fragments, so scripts with more than 0.5%
  damaged words are left out (`clean.junk_share`), and a check below puts every script back with the damage share
  as a control.
- Sequels. The first review word list was "installment", "prequel", "predecessor". Reviews say when a film is a
  sequel, and sequels return more than their budget predicts. The franchise flag (Wikidata) is now a control in
  the primary specification; the version without it is reported next to it.

## 2. The vectors alone know budget, not profit

Every text is cut into 180-word passages (the model reads at most 256 word pieces), each passage is embedded with
all-MiniLM-L6-v2, and a film's vector is the normalised mean. The vectors are reduced to 50 principal components
(unsupervised, so this step never sees the outcome) and a ridge regression predicts the outcome. Every R² is
**out-of-fold**: each film is predicted by a model fitted on the other four fifths, ten times over.

In [3]:
R.fig_alone(r).show()
display(Markdown(say["alone"]))

Audience reviews alone explain 39% of the variation in budget and 27% in revenue on films the model didn't see, but only 11% in profitability. Screenplays alone explain 35% of the variation in budget and 22% in revenue on films the model didn't see, but only 3% in profitability. Plot summaries alone explain 27% of the variation in budget and 20% in revenue on films the model didn't see, but only 2% in profitability. The text knows how big and expensive a film is (blockbusters read like blockbusters) far better than whether it earned its budget back. Revenue is mostly budget, so a correlation with revenue alone says little about profit.

## 3. What the text adds beyond budget, year, genre and franchise

The question that matters for profit: once we know the budget, the year, the genres and whether the film is a
sequel, does the text tell us anything more? The controls are fitted by least squares on the training folds; the
ridge regression on the text predicts what they leave. **The null** gives each film the vector of another film
from its own genre × decade cell, 500 times, which breaks the link between text and money and keeps everything else.
With budget controlled, profitability and revenue are the same question: ln(gross) = ln(gross/budget) + ln(budget).

In [4]:
R.fig_net(res).show()
display(R.main_table(r))
display(Markdown(say["net"]))

,corpus,text,films,alone: R² profitability,alone: R² revenue,alone: R² budget,"gain, no franchise control",R² controls only,R² gain from text,gain 95% interval,shuffle 95th pct,shuffle p,partial Spearman
0,Audience reviews,raw,5405,0.103,0.290,0.389,0.067,0.183,0.040,+0.032 to +0.049,0.001,0.002,0.233
1,Audience reviews,clean,5405,0.111,0.269,0.387,0.073,0.183,0.044,+0.035 to +0.053,0.001,0.002,0.248
2,Screenplays,raw,1538,0.011,0.163,0.226,0.005,0.193,0.003,-0.003 to +0.006,0.002,0.024,0.071
3,Screenplays,clean,1538,0.026,0.218,0.346,0.008,0.193,0.003,+0.002 to +0.010,0.001,0.014,0.072
4,Plot summaries,raw,1625,0.012,0.196,0.260,0.008,0.198,0.005,-0.001 to +0.011,0.002,0.004,0.085
5,Plot summaries,clean,1625,0.018,0.203,0.273,0.014,0.198,0.007,+0.003 to +0.018,0.001,0.002,0.105


Audience reviews (5,405 films) add +0.044 to the R² of ln(gross / budget) (95% interval +0.035 to +0.053; shuffle 95th percentile +0.001, shuffle p = 0.002), against +0.073 without the franchise control. Screenplays (1,538 films) add +0.003 to the R² of ln(gross / budget) (95% interval +0.002 to +0.010; shuffle 95th percentile +0.001, shuffle p = 0.014), against +0.008 without the franchise control. Plot summaries (1,625 films) add +0.007 to the R² of ln(gross / budget) (95% interval +0.003 to +0.018; shuffle 95th percentile +0.001, shuffle p = 0.002), against +0.014 without the franchise control.

**What this measures and what it misses.** The R² gain measures how much a linear map from the text's 50
main directions improves the prediction of profit on unseen films. It captures what a general-purpose sentence model
encodes (topic, tone, genre markers, register); it misses anything the model doesn't encode and any non-linear
pattern. For reviews it is not a forecast: reviews are written after release, and even with money sentences removed,
"everyone saw this" or "nobody talks about it" survives. It says what audiences write about films that did better
than their budget predicted. Screenplays and summaries are the before-release question.

## 4. Checks: a second method, later films, and the rating

- **TF-IDF** is a different representation of the same cleaned text: word counts weighted by rarity, reduced to
  300 dimensions. If it agrees, the result isn't a quirk of one model.
- **Time split**: fitted on the earliest 70% of films and scored on the rest, the only version that could have been
  used in advance (year squared is dropped here, because fitted on early years it extrapolates wildly).
- **Rating**: is the text more than "people liked it"? The IMDb rating joins the controls.
- **Raw vs clean text**: in the table in Section 3. Cleaning should not *create* signal; if raw text scores higher,
  the extra came from formatting, names or money talk.

In [5]:
R.fig_methods(r).show()
display(R.checks_table(r))
display(Markdown(say["checks"]))

,corpus,gain (5-fold),"gain, TF-IDF instead",TF-IDF shuffle p,"gain, later films only",later films,trained up to,gain beyond IMDb rating,rating shuffle p,network assortativity,assortativity null 95th,assortativity p
0,Audience reviews,0.044,0.105,0.005,0.007,1591,2009,0.041,0.005,0.073,0.009,0.002
1,Screenplays,0.003,0.007,0.005,0.002,413,2013,0.011,0.005,0.031,0.013,0.002
2,Plot summaries,0.007,0.004,0.005,0.011,487,2012,0.009,0.005,0.026,0.013,0.002


Audience reviews: TF-IDF gives +0.105 (shuffle p = 0.005); trained on films up to 2009 and scored on the 1,591 later ones, +0.007; with the IMDb rating also controlled, +0.041 (shuffle p = 0.005). Screenplays: TF-IDF gives +0.007 (shuffle p = 0.005); trained on films up to 2013 and scored on the 413 later ones, +0.002; with the IMDb rating also controlled, +0.011 (shuffle p = 0.005). Plot summaries: TF-IDF gives +0.004 (shuffle p = 0.005); trained on films up to 2012 and scored on the 487 later ones, +0.011; with the IMDb rating also controlled, +0.009 (shuffle p = 0.005).

In [6]:
display(R.ocr_table(r))
display(Markdown(say.get("ocr", "")))

,representation,scripts,damaged (above cutoff),Spearman: damage vs profit left by controls,p,gain with damage as control,shuffle p
0,embedding,1630,92,0.003,0.905,0.003,0.015
1,tfidf,1630,92,0.003,0.905,0.007,0.005


92 of 1,630 scripts have more than 0.5% OCR-damaged words. Damage correlates with the profit left by the controls at ρ = +0.003 (p = 0.905). On every script with damage as a control the screenplay gain is +0.003 (shuffle p = 0.015).

## 5. Other samples and the number of components

The 100k+ tier has hand-checked money but few flops; dropping franchise films removes the sequel effect entirely
instead of controlling for it; the shared sample puts the three corpora on the same films.

In [7]:
R.fig_samples(r).show()
display(R.samples_table(r))
display(Markdown(say["samples"]))
display(R.components_table(r))

,corpus,sample,films,R² controls only,R² gain from text,shuffle 95th pct,shuffle p,partial Spearman
0,Audience reviews,all clean films,5405,0.183,0.044,0.001,0.002,0.248
1,Audience reviews,100k+ votes only,1969,0.290,0.031,0.001,0.005,0.197
2,Audience reviews,no franchise films,3979,0.083,0.047,0.001,0.005,0.240
3,Audience reviews,films in all three corpora,1385,0.227,0.031,0.001,0.005,0.208
4,Screenplays,all clean films,1538,0.193,0.003,0.001,0.014,0.072
5,Screenplays,100k+ votes only,1007,0.249,0.006,0.002,0.010,0.089
6,Screenplays,no franchise films,1080,0.052,-0.002,0.001,0.687,0.019
7,Screenplays,films in all three corpora,1301,0.219,0.001,0.002,0.070,0.058
8,Screenplays,"every script, OCR damage as a control",1630,NaN,0.003,0.001,0.015,NaN
9,Plot summaries,all clean films,1625,0.198,0.007,0.001,0.002,0.105


Audience reviews without franchise films: +0.047 (3,979 films, shuffle p = 0.005). Screenplays without franchise films: -0.002 (1,080 films, shuffle p = 0.687). Plot summaries without franchise films: +0.005 (1,129 films, shuffle p = 0.010).

corpus,components,Audience reviews,Screenplays,Plot summaries
0,10,0.002,0.006,-0.001
1,20,0.029,0.005,0.002
2,50,0.044,0.003,0.007
3,100,0.066,0.003,0.008


## 6. Text as a network

The course's second way to join text and networks: link each film to its 10 nearest films by cosine similarity of
the vectors, and ask whether linked films have similar outcomes (Newman's assortativity, the correlation of the
outcome at the two ends of an edge). The null shuffles the outcome across films, inside genre × decade cells for the
residual. This is a check on the regression from a different angle: it uses all 384 dimensions and no fitting.

In [8]:
R.fig_network(r).show()
display(Markdown(say["network"]))
display(Markdown(say["agreement"]))

On the audience reviews network (45,047 edges) the assortativity of profit left by the controls is 0.073 against a shuffle 95th percentile of 0.009 (z = 14.3). On the screenplays network (12,786 edges) the assortativity of profit left by the controls is 0.031 against a shuffle 95th percentile of 0.013 (z = 3.7). On the plot summaries network (14,172 edges) the assortativity of profit left by the controls is 0.026 against a shuffle 95th percentile of 0.013 (z = 3.3).

On the 1,301 films with all three texts, the per-film 'more profit than expected' scores of the three corpora correlate at reviews and screenplays ρ = 0.24, reviews and summaries ρ = 0.22, screenplays and summaries ρ = 0.35. The corpora partly point at the same films.

In [ ]:
# Local only: needs data/raw and the private film table. Hover a point for the film.
for corpus in R.CORPORA:
    d = R.film_map(corpus)
    if d is None:
        print("No local vectors; run s01 and s02 to see the map.")
        break
    R.fig_map(d, corpus).show()

## 7. Which words go with the direction

For each corpus, films are ranked by their out-of-fold "more profit than the controls predict" score from the text.
The words below are used by a larger share of films in the top fifth than in the bottom fifth (and the reverse). They
describe the direction the regression found; they are not causes.

In [10]:
for c in R.CORPORA:
    print(R.LABEL[c])
    display(R.words_table(res["words"], c))

Audience reviews


,"more profit than budget, year, genre and franchise predict",less profit than they predict
0,tearjerker,wilderness
1,tissues,apocalyptic
2,jerker,artsy
3,warming,westerns
4,jerking,futuristic
5,heartwarming,undead
6,daughters,swords
7,cried,auteur
8,teachers,inhabitants
9,marriages,retro


Screenplays


,"more profit than budget, year, genre and franchise predict",less profit than they predict
0,nudging,blacksmith
1,leisurely,pornography
2,favourite,orgy
3,inclination,earnings
4,takeoff,palette
5,mayday,cuban
6,cope,forge
7,bannister,pummeling
8,ports,glimmering
9,decipher,producers


Plot summaries


,"more profit than budget, year, genre and franchise predict",less profit than they predict
0,letters,trial
1,airport,kids
2,traps,manhattan
3,intelligence,character
4,base,writer
5,senator,favor
6,proposes,beautiful
7,bedroom,victim
8,fighter,coast
9,hidden,ambulance


## 8. What to take from this

The numbers are in the readings above, generated from the saved outputs.

- **Reviews** carry a real signal about profit beyond budget, year, genre and franchise. It passes the shuffle null,
  a second method (TF-IDF finds more, not less), a control for the IMDb rating, and the no-franchise sample. It is
  not a forecast: the reviews are written after release, and fitted on films up to 2009 the gain on later films is
  much smaller, so part of what the model learns is period-specific. The words point at audience-pleasing drama
  ("tearjerker", "heartwarming") doing better than its budget predicts and at genre fare (westerns, apocalyptic
  and futuristic films, "artsy") doing worse.
- **Plot summaries** add a small amount that survives without franchise films.
- **Screenplays** add almost nothing: the small gain on all films vanishes once franchise films are left out, so
  what the scripts carried was mostly whether the film is a sequel. Cleaning matters here: before the OCR filter,
  damaged scans made the screenplay word list.
- **Budget, not profit**: every text predicts budget and revenue far better than profitability. A correlation of
  text with revenue mostly says big films read like big films.

Limits: MovieSum covers mostly well-known films, which skews towards hits, and its scripts mix drafts; name masking
is incomplete (plural surnames such as "the Coens", other films' titles such as "Shawshank" inside reviews); the
franchise flag is noisy; the OCR cutoff and the franchise control were added after the first results; gross over
budget isn't profit.